In [1]:
import pandas as pd
import numpy as np

In [2]:
train = pd.read_csv('/Users/deannalakshman/Desktop/IIT/YEAR 2/SEM 2/DSPLC/Coursework technical/EDA/cleaned_dataset/train_cleaned.csv')
val   = pd.read_csv('/Users/deannalakshman/Desktop/IIT/YEAR 2/SEM 2/DSPLC/Coursework technical/EDA/cleaned_dataset/val_cleaned.csv')
test  = pd.read_csv('/Users/deannalakshman/Desktop/IIT/YEAR 2/SEM 2/DSPLC/Coursework technical/EDA/cleaned_dataset/test_cleaned.csv')

print('Train shape     :', train.shape)
print('Validation shape:', val.shape)
print('Test shape      :', test.shape)

Train shape     : (27499, 27)
Validation shape: (2749, 27)
Test shape      : (4318, 23)


In [3]:
#convert data columns

date_cols = ['Expected_checkin', 'Expected_checkout', 'Booking_date']

for df in [train, val, test]:
    for col in date_cols:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# Verify
print(train[date_cols].dtypes)
print(val[date_cols].dtypes)

Expected_checkin     datetime64[ns]
Expected_checkout    datetime64[ns]
Booking_date         datetime64[ns]
dtype: object
Expected_checkin     datetime64[ns]
Expected_checkout    datetime64[ns]
Booking_date         datetime64[ns]
dtype: object


In [4]:
#lead time - days between booking and expected check-in

for df in [train, val, test]:
    df['Lead_Time'] = (df['Expected_checkin'] - df['Booking_date']).dt.days

# Verify
print('Lead Time stats:')
print(train['Lead_Time'].describe())

Lead Time stats:
count    27499.000000
mean       109.910906
std         78.048838
min         -4.000000
25%         45.000000
50%        101.000000
75%        166.000000
max        708.000000
Name: Lead_Time, dtype: float64


In [5]:
# stay duration - days between expected check-in and expected check-out

for df in [train, val, test]:
    df['Stay_Duration'] = (df['Expected_checkout'] - df['Expected_checkin']).dt.days

# Verify
print('Stay Duration stats:')
print(train['Stay_Duration'].describe())

Stay Duration stats:
count    27499.000000
mean         1.836212
std          0.985204
min          1.000000
25%          1.000000
50%          2.000000
75%          2.000000
max          4.000000
Name: Stay_Duration, dtype: float64


In [6]:
# Booking month and checking month

for df in [train, val, test]:
    df['Booking_Month']  = df['Booking_date'].dt.month
    df['Checkin_Month']  = df['Expected_checkin'].dt.month

# Verify
print('Booking Month value counts:')
print(train['Booking_Month'].value_counts().sort_index())

print('\nCheckin Month value counts:')
print(train['Checkin_Month'].value_counts().sort_index())

Booking Month value counts:
Booking_Month
1     2265
2     2164
3     1969
4     2061
5     2134
6     2209
7     2366
8     2267
9     2283
10    3247
11    2352
12    2182
Name: count, dtype: int64

Checkin Month value counts:
Checkin_Month
1     2372
2     2237
3     2580
4     2395
5     2256
6     2139
7     3002
8     2978
9     1840
10    2250
11    1743
12    1707
Name: count, dtype: int64


In [7]:
# check in day of week

# Monday=0, Sunday=6
for df in [train, val, test]:
    df['Checkin_DayOfWeek'] = df['Expected_checkin'].dt.dayofweek
    df['Is_Weekend_Checkin'] = df['Checkin_DayOfWeek'].apply(lambda x: 1 if x >= 5 else 0)

# Verify
print('Day of Week value counts (0=Mon, 6=Sun):')
print(train['Checkin_DayOfWeek'].value_counts().sort_index())

print('\nWeekend Check-ins:')
print(train['Is_Weekend_Checkin'].value_counts())

Day of Week value counts (0=Mon, 6=Sun):
Checkin_DayOfWeek
0    4460
1    4087
2    3959
3    4109
4    4062
5    3024
6    3798
Name: count, dtype: int64

Weekend Check-ins:
Is_Weekend_Checkin
0    20677
1     6822
Name: count, dtype: int64


In [8]:
# total guests

for df in [train, val, test]:
    df['Total_Guests'] = df['Adults'] + df['Children'] + df['Babies']

# Verify
print('Total Guests stats:')
print(train['Total_Guests'].describe())
print('\nTotal Guests value counts:')
print(train['Total_Guests'].value_counts().sort_index())

Total Guests stats:
count    27499.000000
mean         4.153096
std          1.120349
min          2.000000
25%          3.000000
50%          4.000000
75%          5.000000
max          7.000000
Name: Total_Guests, dtype: float64

Total Guests value counts:
Total_Guests
2.0    2009
3.0    6047
4.0    7654
4.5    1591
5.0    7027
5.5     589
6.0    2124
6.5     106
7.0     352
Name: count, dtype: int64


In [9]:
# has children and has babies flags
for df in [train, val, test]:
    df['Has_Children'] = (df['Children'] > 0).astype(int)
    df['Has_Babies']   = (df['Babies'] > 0).astype(int)

# Verify
print('Has Children:')
print(train['Has_Children'].value_counts())

print('\nHas Babies:')
print(train['Has_Babies'].value_counts())

Has Children:
Has_Children
1    25213
0     2286
Name: count, dtype: int64

Has Babies:
Has_Babies
0    19217
1     8282
Name: count, dtype: int64


In [10]:
# Net Room Rate after discount

for df in [train, val, test]:
    df['Net_Room_Rate'] = df['Room_Rate'] * (1 - df['Discount_Rate'] / 100)

# Verify
print('Net Room Rate stats:')
print(train['Net_Room_Rate'].describe())

Net Room Rate stats:
count    27499.000000
mean       153.273508
std         43.469622
min         60.000000
25%        117.600000
50%        150.300000
75%        186.750000
max        250.000000
Name: Net_Room_Rate, dtype: float64


In [11]:
# check if all the features have been created successfully

new_features = ['Lead_Time', 'Stay_Duration', 'Booking_Month', 'Checkin_Month',
                'Checkin_DayOfWeek', 'Is_Weekend_Checkin', 'Total_Guests',
                'Has_Children', 'Has_Babies', 'Net_Room_Rate']

for name, df in [('TRAIN', train), ('VALIDATION', val), ('TEST', test)]:
    print(f'\n{name}')
    for feat in new_features:
        missing = df[feat].isnull().sum()
        print(f'  {feat:<25} — created: {"Yes" if feat in df.columns else "No"}   missing: {missing}')


TRAIN
  Lead_Time                 — created: Yes   missing: 0
  Stay_Duration             — created: Yes   missing: 0
  Booking_Month             — created: Yes   missing: 0
  Checkin_Month             — created: Yes   missing: 0
  Checkin_DayOfWeek         — created: Yes   missing: 0
  Is_Weekend_Checkin        — created: Yes   missing: 0
  Total_Guests              — created: Yes   missing: 0
  Has_Children              — created: Yes   missing: 0
  Has_Babies                — created: Yes   missing: 0
  Net_Room_Rate             — created: Yes   missing: 0

VALIDATION
  Lead_Time                 — created: Yes   missing: 0
  Stay_Duration             — created: Yes   missing: 0
  Booking_Month             — created: Yes   missing: 0
  Checkin_Month             — created: Yes   missing: 0
  Checkin_DayOfWeek         — created: Yes   missing: 0
  Is_Weekend_Checkin        — created: Yes   missing: 0
  Total_Guests              — created: Yes   missing: 0
  Has_Children              —

In [12]:
import os

os.makedirs('featured_dataset', exist_ok=True)

train.to_csv('featured_dataset/train_featured.csv', index=False)
val.to_csv('featured_dataset/val_featured.csv',     index=False)
test.to_csv('featured_dataset/test_featured.csv',   index=False)

print('Feature engineered datasets saved to featured_dataset folder.')

Feature engineered datasets saved to featured_dataset folder.
